In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

transaction_train_df = pd.read_csv("raw_files/transactions_train.csv")
transaction_train_df

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.051,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.017,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.017,2
...,...,...,...,...,...
31788319,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,929511001,0.059,2
31788320,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,891322004,0.042,2
31788321,2020-09-22,fff380805474b287b05cb2a7507b9a013482f7dd0bce0e...,918325001,0.043,1
31788322,2020-09-22,fff4d3a8b1f3b60af93e78c30a7cb4cf75edaf2590d3e5...,833459002,0.007,1


In [8]:
transaction_train_df["t_dat"] = pd.to_datetime(transaction_train_df["t_dat"])

print("Rows:", len(transaction_train_df))
print("Unique customers:", transaction_train_df["customer_id"].nunique())
print("Date range:", transaction_train_df["t_dat"].min().date(), "-", transaction_train_df["t_dat"].max().date())

Rows: 31788324
Unique customers: 1362281
Date range: 2018-09-20 - 2020-09-22


Находим первую покупку клиента

In [9]:
first_purchase = (transaction_train_df.groupby("customer_id", as_index=False)["t_dat"].min()
                  .rename(columns = {"t_dat": "first_purchase_date"}))
first_purchase.head()

,customer_id,first_purchase_date
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2018-12-27
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,2018-09-21
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2018-09-20
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,2019-06-09
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,2018-10-12


Присоединяем дату первой покупки к каждой транзакции

In [10]:
tx = transaction_train_df.merge(first_purchase, on = "customer_id", how = "left")
tx.head()

,t_dat,customer_id,article_id,price,sales_channel_id,first_purchase_date
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.051,2,2018-09-20
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030,2,2018-09-20
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015,2,2018-09-20
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.017,2,2018-09-20
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.017,2,2018-09-20


Формируем когорту по месяцу первой покупки

In [12]:
tx["cohort_month"] = tx["first_purchase_date"].dt.strftime("%Y-%m")
tx[["customer_id", "t_dat", "first_purchase_date", "cohort_month"]].head()

,customer_id,t_dat,first_purchase_date,cohort_month
0,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2018-09-20,2018-09-20,2018-09
1,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2018-09-20,2018-09-20,2018-09
2,00007d2de826758b65a93dd24ce629ed66842531df6699...,2018-09-20,2018-09-20,2018-09
3,00007d2de826758b65a93dd24ce629ed66842531df6699...,2018-09-20,2018-09-20,2018-09
4,00007d2de826758b65a93dd24ce629ed66842531df6699...,2018-09-20,2018-09-20,2018-09


Считаем, сколько дней прошло с первой покупки для каждой транзакции

In [13]:
tx["days_since_first_purchase"] = (tx["t_dat"] - tx["first_purchase_date"]).dt.days
tx["days_since_first_purchase"].describe()

count   31788324.000
mean         260.292
std          214.204
min            0.000
25%           58.000
50%          231.000
75%          429.000
max          733.000
Name: days_since_first_purchase, dtype: float64

Смотрим, сколько клиентов в каждом месяце

In [ ]:
cohort_sizes = (
    tx.groupby("cohort_month")["customer_id"]
    .nunique()
    .sort_index()
    .rename("cohort_size")
    .reset_index()
)
cohort_sizes.head()

,cohort_month,cohort_size
0,2018-09,140340
1,2018-10,212669
2,2018-11,138233
3,2018-12,89944
4,2019-01,68957


In [ ]:
cohort_sizes["cohort_size"].describe()

count       25.000
mean     54491.240
std      46080.463
min      17538.000
25%      28242.000
50%      36990.000
75%      52127.000
max     212669.000
Name: cohort_size, dtype: float64

Когорты по месяцу сильно отличаются, возможно это влияют неполные месяц в начале или конце периода. Проверим неполные месяцы.

In [17]:
calendar_coverage = (
    tx.assign(cal_month = tx["t_dat"].dt.to_period("M"))
      .groupby("cal_month")["t_dat"]
      .agg(min_date="min", max_date="max", n_days=lambda s: s.dt.date.nunique())
      .reset_index()
)
calendar_coverage.head()

,cal_month,min_date,max_date,n_days
0,2018-09,2018-09-20,2018-09-30,11
1,2018-10,2018-10-01,2018-10-31,31
2,2018-11,2018-11-01,2018-11-30,30
3,2018-12,2018-12-01,2018-12-31,31
4,2019-01,2019-01-01,2019-01-31,31


Убираем неполные крайние месяцы

In [23]:
max_date = tx["t_dat"].max()
cohort_start = (tx.groupby("cohort_month")["first_purchase_date"].min()) 

eligible_cohorts_90 = cohort_start[cohort_start <= (max_date - pd.Timedelta(days=90))].index

tx_eligible_90 = tx[tx["cohort_month"].isin(eligible_cohorts_90)].copy()

print("Когорты в целом:", tx["cohort_month"].nunique())
print("Подходящие когорты день 90:", tx_eligible_90["cohort_month"].nunique())


Когорты в целом: 25
Подходящие когорты день 90: 22


Для расчёта удержания на 90-й день используем только те когорты, для которых доступен полный период наблюдения. Когорты, сформированные ближе к концу временного диапазона данных, исключены из анализа, так как отсутствие покупок на 90-й день в этом случае связано с ограничением данных, а не с поведением клиентов.

In [ ]:
tx_cohort = tx[[
        "customer_id", "t_dat",
        "first_purchase_date", "cohort_month",
        "days_since_first_purchase"
    ]].copy()

tx_cohort.to_csv("processed_files/transactions_with_cohorts.csv", index=False)

: 

Когорты определены по месяцу первой покупки.
Для каждой транзакции рассчитано число дней с первой покупки, чтобы посчитать удержание на 7/30/90 день.
Подготовлена база для расчёта retention и визуализации heatmap.